**G1-3:** <br>
Daniel Cruz Flores (1709912) <br>
Marco Mejías Alés (1710748) <br>
Marc Solés i Rojas (1710741) <br>
Samuel Jesús Outeda Aponte (1711378)

Si tenemos una imagen obtenida de una cámara de 61MP, con una profundidad de color de 16 bpp, podemos calcular el tamaño resultante de la siguiente forma:

$$
61MP \cdot 16 bpp \cdot \frac{1 byte}{8 bits} = 122 MB
$$

Esto asumiendo que no se usa ningún tipo de compresión.

In [ ]:
import os
import cv2
import numpy as np
import metrikz
import matplotlib.pyplot as plt
from PIL import Image

### **Pregunta 1**

Si tenemos una imagen obtenida de una cámara de 61MP, con una profundidad de color de 16 bpp, podemos calcular el tamaño resultante de la siguiente forma:

$$
61MP \cdot 16 bpp \cdot \frac{1 byte}{8 bits} = 122 MB
$$

Esto asumiendo que no se usa ningún tipo de compresión.

In [ ]:
# Directorios de entrada y salida
DATASET_DIR = "dataset"
OUTPUT_DIR = "output"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def main():
    images = [f for f in os.listdir(DATASET_DIR) if f.lower().endswith('.png')]
    qualities = [1] + list(range(10, 101, 10))
    
    mse_results = {img: [] for img in images}
    ratio_results = {img: [] for img in images}

    print("--- Análisis JPEG (OpenCV + metrikz) ---")
    for img_name in images:
        orig_path = os.path.join(DATASET_DIR, img_name)
        
        # Leer imagen PNG (BGR por defecto en OpenCV)
        source = cv2.imread(orig_path, cv2.IMREAD_COLOR)
        if source is None:
            print(f"Error: No se pudo cargar {img_name}")
            continue
            
        orig_size = os.path.getsize(orig_path)
        source_np = np.asarray(source)
        
        for q in qualities:
            jpeg_path = os.path.join(OUTPUT_DIR, f"{os.path.splitext(img_name)[0]}_q{q}.jpg")
            
            # Guardar como JPEG con la calidad iterada [1-100]
            cv2.imwrite(jpeg_path, source, [int(cv2.IMWRITE_JPEG_QUALITY), q])
            
            # Leer la imagen JPEG resultante
            target = cv2.imread(jpeg_path, cv2.IMREAD_COLOR)
            target_np = np.asarray(target)
            
            # Calcular MSE utilizando la librería metrikz
            mse = metrikz.mse(source_np, target_np)
            mse_results[img_name].append(mse)
            
            # Calcular Ratio de compresión: original / resultante
            jpeg_size = os.path.getsize(jpeg_path)
            ratio = orig_size / jpeg_size
            ratio_results[img_name].append(ratio)

    # Generar Figura 1: MSE vs Quality
    plt.figure(1)
    for img_name in images:
        plt.plot(qualities, mse_results[img_name], marker='o', label=img_name)
    plt.xlabel('Quality')
    plt.ylabel('MSE')
    plt.title('Figura 1: MSE vs Quality')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Figura_1_MSE.png"))
    
    # Generar Figura 2: Ratio compresión vs Quality
    plt.figure(2)
    for img_name in images:
        plt.plot(qualities, ratio_results[img_name], marker='x', label=img_name)
    plt.xlabel('Quality')
    plt.ylabel('Ràtio Compressió')
    plt.title('Figura 2: Relació de compressió vs Quality')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "Figura_2_Ratio.png"))
    print("Gráficas guardadas correctamente.\n")

    print("--- Análisis GIF (PIL) ---")
    # Utilizamos PIL porque OpenCV no maneja escritura de GIFs
    for img_name in images:
        orig_path = os.path.join(DATASET_DIR, img_name)
        img_orig = Image.open(orig_path)
        base_name = os.path.splitext(img_name)[0]
        gif_path = os.path.join(OUTPUT_DIR, f"{base_name}.gif")
        
        # Convertir a GIF
        img_gif = img_orig.convert("P", palette=Image.ADAPTIVE)
        img_gif.save(gif_path, "GIF")
        
        print(f"\nAnálisis de {img_name}:")
        
        # Colores en el GIF
        colors_gif = img_gif.getcolors(maxcolors=256)
        if colors_gif:
            print(f"  [GIF] Total colores diferentes: {len(colors_gif)}")
            sorted_gif = sorted(colors_gif, key=lambda x: x[0], reverse=True)
            print("  [GIF] Top 3 colores más frecuentes:", sorted_gif[:3])
            
        # Colores en el PNG original
        colors_png = img_orig.getcolors(maxcolors=16777216)
        if colors_png:
            print(f"  [PNG original] Total colores diferentes: {len(colors_png)}")
            sorted_png = sorted(colors_png, key=lambda x: x[0], reverse=True)
            print("  [PNG original] Top 3 colores más frecuentes (Conteo, RGB):", sorted_png[:3])

        # Cambiar la paleta para que la componente roja sea 0 en imágenes específicas
        if base_name in ["3", "7", "8"]:
            palette = img_gif.getpalette() # Devuelve [R, G, B, R, G, B...] 
            if palette:
                # Modificar componente R a 0
                for i in range(0, len(palette), 3):
                    palette[i] = 0 
                
                img_gif.putpalette(palette)
                mod_gif_path = os.path.join(OUTPUT_DIR, f"{base_name}_R0.gif")
                img_gif.save(mod_gif_path, "GIF")
                print(f"Paleta modificada (R=0) guardada como {base_name}_R0.gif.")

if __name__ == "__main__":
    main()